# Natural Language to SQL — Schema Grounding

Text-to-SQL fails most often when the model **does not know your schema** — it invents column names, joins the wrong tables, or misses foreign-key paths. **Schema grounding** is the practice of feeding accurate, scoped database metadata into the NL→SQL step so generated queries reference **real** tables and columns.

```
User question
     │
     ▼
┌─────────────┐     ┌──────────────┐     ┌─────────────┐
│ Schema      │ ──► │ Table/column │ ──► │ SQL         │
│ retrieval   │     │ linking      │     │ generation  │
└─────────────┘     └──────────────┘     └─────────────┘
     │                                         │
     └──────── sample values, FK docs ─────────┘
```

This notebook extends **ShopCo** with production-shaped schema grounding techniques:

1. Multiple **schema views** (compact, rich, DDL)
2. **Schema linking** — map user phrases to tables/columns
3. **Pruning** — send only relevant tables to the planner
4. **Few-shot examples** anchored to the schema
5. **Validation** — reject SQL referencing unknown identifiers
6. **Eval harness** — measure grounding quality

Self-contained SQLite + plain Python. Optional live LLM cell at the end.

In [ ]:
"""
ShopCo Text-to-SQL — schema grounding lab.
"""

import json
import os
import re
import sqlite3
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def create_shopco_db() -> sqlite3.Connection:
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.executescript("""
        CREATE TABLE customers (
            customer_id INTEGER PRIMARY KEY,
            email TEXT NOT NULL UNIQUE,
            name TEXT NOT NULL,
            tier TEXT DEFAULT 'standard'
        );
        CREATE TABLE products (
            product_id INTEGER PRIMARY KEY,
            sku TEXT NOT NULL UNIQUE,
            name TEXT NOT NULL,
            price_usd REAL NOT NULL
        );
        CREATE TABLE orders (
            order_id TEXT PRIMARY KEY,
            customer_id INTEGER NOT NULL,
            status TEXT NOT NULL,
            total_usd REAL NOT NULL,
            tracking TEXT,
            created_at TEXT NOT NULL
        );
        CREATE TABLE order_items (
            item_id INTEGER PRIMARY KEY,
            order_id TEXT NOT NULL,
            product_id INTEGER NOT NULL,
            quantity INTEGER NOT NULL
        );
    """)
    conn.executemany("INSERT INTO customers VALUES (?, ?, ?, ?)", [
        (1, "alice@shopco.com", "Alice Nguyen", "gold"),
        (2, "bob@shopco.com", "Bob Smith", "standard"),
        (3, "carol@shopco.com", "Carol Lee", "standard"),
    ])
    conn.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", [
        (1, "MOUSE-01", "Wireless Mouse", 29.99),
        (2, "HUB-02", "USB-C Hub", 49.99),
        (3, "LAMP-03", "Desk Lamp", 42.00),
    ])
    conn.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?)", [
        ("ORD-1001", 1, "shipped", 89.50, "1Z999AA10123456784", "2026-07-01"),
        ("ORD-1002", 2, "processing", 42.00, None, "2026-07-08"),
        ("ORD-1003", 3, "delivered", 120.00, "1Z999AA10987654321", "2026-06-15"),
    ])
    conn.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?)", [
        (1, "ORD-1001", 1, 1), (2, "ORD-1001", 2, 1), (3, "ORD-1002", 3, 1), (4, "ORD-1003", 2, 2),
    ])
    conn.commit()
    return conn


DB = create_shopco_db()
print("ShopCo DB ready")

---

## 1. What Is Schema Grounding?

| Without grounding | With grounding |
|-------------------|----------------|
| `SELECT user_name FROM purchases` | `SELECT name FROM customers` |
| Join on guessed keys | `orders.customer_id = customers.customer_id` |
| Filter `status = 'complete'` | `status = 'delivered'` (real enum values) |
| 200-table prompt overflow | Pruned 3-table subset for this question |

**Schema grounding** = ensuring the NL→SQL model sees the **right** metadata before it writes SQL.

---

## 2. Rich Schema Registry

Go beyond column types — add **synonyms**, **sample values**, and **join hints**.

In [ ]:
SCHEMA_REGISTRY: dict[str, dict[str, Any]] = {
    "customers": {
        "description": "Customer accounts and loyalty tier",
        "synonyms": ["users", "accounts", "buyers"],
        "columns": {
            "customer_id": {"type": "INTEGER", "pk": True, "desc": "primary key"},
            "email": {"type": "TEXT", "desc": "unique email", "samples": ["alice@shopco.com"]},
            "name": {"type": "TEXT", "desc": "full name"},
            "tier": {"type": "TEXT", "desc": "loyalty tier", "samples": ["standard", "gold"]},
        },
    },
    "products": {
        "description": "Product catalog SKUs",
        "synonyms": ["catalog", "items", "skus"],
        "columns": {
            "product_id": {"type": "INTEGER", "pk": True},
            "sku": {"type": "TEXT", "desc": "SKU code", "samples": ["MOUSE-01"]},
            "name": {"type": "TEXT", "desc": "product name", "samples": ["Desk Lamp"]},
            "price_usd": {"type": "REAL", "desc": "unit price USD"},
        },
    },
    "orders": {
        "description": "Order headers — order_id format ORD-XXXX",
        "synonyms": ["purchases", "transactions"],
        "columns": {
            "order_id": {"type": "TEXT", "pk": True, "samples": ["ORD-1001"]},
            "customer_id": {"type": "INTEGER", "fk": "customers.customer_id"},
            "status": {"type": "TEXT", "samples": ["processing", "shipped", "delivered"]},
            "total_usd": {"type": "REAL"},
            "tracking": {"type": "TEXT", "nullable": True},
            "created_at": {"type": "TEXT"},
        },
    },
    "order_items": {
        "description": "Line items per order",
        "synonyms": ["line items", "order lines"],
        "columns": {
            "item_id": {"type": "INTEGER", "pk": True},
            "order_id": {"type": "TEXT", "fk": "orders.order_id"},
            "product_id": {"type": "INTEGER", "fk": "products.product_id"},
            "quantity": {"type": "INTEGER"},
        },
    },
}

FOREIGN_KEYS = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
]

print("Tables:", list(SCHEMA_REGISTRY.keys()))

---

## 3. Schema View Formats

Different planners need different shapes — compact for routing, rich for generation.

In [ ]:
def schema_compact(registry: dict[str, dict[str, Any]]) -> str:
    lines = []
    for table, meta in registry.items():
        cols = ", ".join(meta["columns"].keys())
        lines.append(f"{table}({cols})")
    return "\n".join(lines)


def schema_rich(registry: dict[str, dict[str, Any]], tables: list[str] | None = None) -> str:
    chosen = tables or list(registry.keys())
    lines = ["# ShopCo Schema (grounded view)"]
    for table in chosen:
        meta = registry[table]
        lines.append(f"\n## {table} — {meta['description']}")
        if meta.get("synonyms"):
            lines.append(f"Synonyms: {', '.join(meta['synonyms'])}")
        for col, spec in meta["columns"].items():
            extra = []
            if spec.get("fk"):
                extra.append(f"FK→{spec['fk']}")
            if spec.get("samples"):
                extra.append(f"e.g. {spec['samples'][:2]}")
            suffix = f" ({', '.join(extra)})" if extra else ""
            lines.append(f"  - {col} {spec.get('type', '')}{suffix}")
    return "\n".join(lines)


def schema_ddl(conn: sqlite3.Connection) -> str:
    rows = conn.execute(
        "SELECT sql FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
    ).fetchall()
    return "\n\n".join(r["sql"] for r in rows if r["sql"])


print("=== Compact ===")
print(schema_compact(SCHEMA_REGISTRY))
print("\n=== Rich (orders only) ===")
print(schema_rich(SCHEMA_REGISTRY, ["orders", "customers"])[:350], "...")

---

## 4. Schema Linking — Map Question → Tables

**Schema linking** selects which tables/columns are relevant before SQL generation — critical for large warehouses.

In [ ]:
LINKING_LEXICON: dict[str, list[str]] = {
    "customers": ["customer", "user", "account", "buyer", "gold", "tier", "email"],
    "products": ["product", "sku", "catalog", "lamp", "mouse", "hub", "item"],
    "orders": ["order", "ord-", "shipped", "processing", "delivered", "tracking", "purchase"],
    "order_items": ["line item", "quantity", "units sold", "revenue"],
}


@dataclass
class SchemaLinkResult:
    tables: list[str]
    columns: list[str]
    confidence: float
    reason: str


def link_schema(question: str, registry: dict[str, dict[str, Any]]) -> SchemaLinkResult:
    q = question.lower()
    table_scores: dict[str, int] = {}
    for table, keywords in LINKING_LEXICON.items():
        score = sum(1 for kw in keywords if kw in q)
        if re.search(r"ORD-[0-9]{4}", question.upper()) and table == "orders":
            score += 3
        if score:
            table_scores[table] = score

    if not table_scores:
        return SchemaLinkResult(["orders"], [], 0.3, "default to orders")

    ranked = sorted(table_scores.items(), key=lambda x: -x[1])
    tables = [t for t, _ in ranked[:3]]

    # Pull FK neighbors so joins are possible
    expanded = set(tables)
    for t in list(expanded):
        for fk in FOREIGN_KEYS:
            if fk[0] in expanded and fk[2] not in expanded:
                expanded.add(fk[2])
            if fk[2] in expanded and fk[0] not in expanded:
                expanded.add(fk[0])
    tables = list(expanded)

    columns: list[str] = []
    for t in tables:
        columns.extend(registry[t]["columns"].keys())

    return SchemaLinkResult(tables, columns, min(1.0, ranked[0][1] / 4), f"matched {ranked[0][0]}")


for q in ["gold tier customers", "revenue from Desk Lamp", "ORD-1001 tracking"]:
    link = link_schema(q, SCHEMA_REGISTRY)
    print(f"Q: {q}")
    print(f"  tables={link.tables} conf={link.confidence:.2f} ({link.reason})")

---

## 5. Few-Shot Examples (Schema-Anchored)

Pair natural language with **valid SQL** using real table/column names — the strongest grounding signal for LLMs.

In [ ]:
FEW_SHOT_EXAMPLES: list[dict[str, str]] = [
    {
        "question": "How many shipped orders?",
        "sql": "SELECT COUNT(*) AS cnt FROM orders WHERE status = 'shipped' LIMIT 100",
    },
    {
        "question": "Status of ORD-1001",
        "sql": "SELECT order_id, status, total_usd, tracking FROM orders WHERE order_id = 'ORD-1001' LIMIT 100",
    },
    {
        "question": "Gold customers and their emails",
        "sql": "SELECT customer_id, name, email FROM customers WHERE tier = 'gold' LIMIT 100",
    },
    {
        "question": "Units sold of Desk Lamp",
        "sql": """
            SELECT p.name, SUM(oi.quantity) AS units
            FROM order_items oi
            JOIN products p ON oi.product_id = p.product_id
            WHERE p.name LIKE '%Desk Lamp%'
            GROUP BY p.name
            LIMIT 100
        """.strip(),
    },
]


def few_shot_prompt_block(examples: list[dict[str, str]], n: int = 3) -> str:
    lines = ["# Examples (question → SQL)"]
    for ex in examples[:n]:
        lines.append(f"Q: {ex['question']}")
        lines.append(f"SQL: {ex['sql']}")
        lines.append("")
    return "\n".join(lines)


print(few_shot_prompt_block(FEW_SHOT_EXAMPLES, 2))

---

## 6. Grounded SQL Planner

Combine **schema linking**, **pruned rich schema**, and **few-shot patterns** in the offline planner.

In [ ]:
@dataclass
class GroundedSQLPlan:
    question: str
    linked_tables: list[str]
    schema_view: str
    sql: str | None
    trace: list[str] = field(default_factory=list)


def grounded_nl_to_sql(question: str, registry: dict[str, dict[str, Any]]) -> GroundedSQLPlan:
    plan = GroundedSQLPlan(question=question, linked_tables=[], schema_view="", sql=None)
    link = link_schema(question, registry)
    plan.linked_tables = link.tables
    plan.schema_view = schema_rich(registry, link.tables)
    plan.trace.append(f"link:{link.tables}")

    q = question.lower()
    m = re.search(r"ORD-[0-9]{4}", question.upper())

    if m:
        plan.sql = f"SELECT order_id, status, total_usd, tracking FROM orders WHERE order_id = '{m.group(0)}'"
    elif "how many" in q and "shipped" in q:
        plan.sql = "SELECT COUNT(*) AS cnt FROM orders WHERE status = 'shipped'"
    elif "gold" in q and "customer" in q:
        plan.sql = "SELECT customer_id, name, email FROM customers WHERE tier = 'gold'"
    elif "lamp" in q or "desk lamp" in q:
        plan.sql = """
            SELECT p.name, SUM(oi.quantity) AS units, SUM(oi.quantity * p.price_usd) AS revenue_usd
            FROM order_items oi
            JOIN products p ON oi.product_id = p.product_id
            WHERE p.name LIKE '%Lamp%'
            GROUP BY p.name
        """.strip()
    elif "highest" in q or "largest" in q:
        plan.sql = "SELECT order_id, total_usd FROM orders ORDER BY total_usd DESC LIMIT 1"

    if plan.sql:
        plan.trace.append("plan:matched")
    else:
        plan.trace.append("plan:no_match")
    return plan


demo_plan = grounded_nl_to_sql("How many shipped orders?", SCHEMA_REGISTRY)
print("Tables:", demo_plan.linked_tables)
print("SQL:", demo_plan.sql)

---

## 7. Identifier Validation — Catch Hallucinated Columns

After SQL generation, verify every `table.column` reference exists in the registry.

In [ ]:
KNOWN_TABLES = set(SCHEMA_REGISTRY.keys())
KNOWN_COLUMNS = {t: set(SCHEMA_REGISTRY[t]["columns"].keys()) for t in KNOWN_TABLES}


def extract_sql_identifiers(sql: str) -> set[str]:
    """Naive extractor for table and column tokens in SQL."""
    tokens = set(re.findall(r"[a-zA-Z_][a-zA-Z0-9_]*", sql))
    return {t.lower() for t in tokens}


def validate_grounded_sql(sql: str, allowed_tables: list[str]) -> list[str]:
    errors: list[str] = []
    if not re.match(r"^\s*SELECT\b", sql, re.IGNORECASE):
        errors.append("not a SELECT statement")
    ids = extract_sql_identifiers(sql)
    sql_keywords = {"select", "from", "where", "join", "on", "and", "or", "as", "sum", "count", "group", "by", "order", "limit", "desc", "asc", "null", "is", "not", "like", "inner", "left", "outer"}
    for token in ids - sql_keywords:
        if token in KNOWN_TABLES:
            continue
        found = any(token in KNOWN_COLUMNS[t] for t in allowed_tables if t in KNOWN_COLUMNS)
        if not found and token not in {"cnt", "units", "revenue_usd", "units_sold"}:
            errors.append(f"unknown identifier: {token}")
    return errors


good_sql = "SELECT order_id, status FROM orders WHERE status = 'shipped'"
bad_sql = "SELECT user_name, purchase_total FROM transactions WHERE state = 'complete'"
print("Good:", validate_grounded_sql(good_sql, ["orders"]))
print("Bad:", validate_grounded_sql(bad_sql, ["orders"]))

---

## 8. Safe Executor

In [ ]:
FORBIDDEN = re.compile(r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE)\b", re.I)


@dataclass
class QueryResult:
    sql: str
    rows: list[dict[str, Any]]
    error: str | None = None


def execute_read_sql(conn: sqlite3.Connection, sql: str) -> QueryResult:
    cleaned = sql.strip().rstrip(";")
    if FORBIDDEN.search(cleaned):
        return QueryResult(sql=sql, rows=[], error="write operations blocked")
    if "limit" not in cleaned.lower():
        cleaned += " LIMIT 100"
    try:
        rows = [dict(r) for r in conn.execute(cleaned).fetchall()]
        return QueryResult(sql=cleaned, rows=rows)
    except sqlite3.Error as exc:
        return QueryResult(sql=cleaned, rows=[], error=str(exc))


print(execute_read_sql(DB, demo_plan.sql or ""))

---

## 9. End-to-End Grounded Text-to-SQL Agent

In [ ]:
@dataclass
class TextToSQLResult:
    question: str
    plan: GroundedSQLPlan
    validation_errors: list[str]
    query: QueryResult | None
    answer: str


class SchemaGroundedTextToSQL:
    def __init__(self, conn: sqlite3.Connection, registry: dict[str, dict[str, Any]]) -> None:
        self.conn = conn
        self.registry = registry

    def run(self, question: str) -> TextToSQLResult:
        plan = grounded_nl_to_sql(question, self.registry)
        if not plan.sql:
            return TextToSQLResult(
                question=question, plan=plan, validation_errors=["no sql planned"],
                query=None, answer="Could not map question to schema.",
            )
        val_errors = validate_grounded_sql(plan.sql, plan.linked_tables)
        if val_errors:
            return TextToSQLResult(
                question=question, plan=plan, validation_errors=val_errors,
                query=None, answer=f"SQL failed grounding check: {val_errors}",
            )
        qr = execute_read_sql(self.conn, plan.sql)
        if qr.error:
            answer = f"Query error: {qr.error}"
        elif not qr.rows:
            answer = "No rows matched."
        else:
            answer = f"Result: {pretty(qr.rows[:3])}\n[SQL] {qr.sql}"
        return TextToSQLResult(question, plan, val_errors, qr, answer)

    def prompt_context(self, question: str) -> str:
        """Full grounding bundle for live LLM."""
        link = link_schema(question, self.registry)
        return "\n\n".join([
            schema_rich(self.registry, link.tables),
            few_shot_prompt_block(FEW_SHOT_EXAMPLES),
            f"# User question\n{question}",
        ])


agent = SchemaGroundedTextToSQL(DB, SCHEMA_REGISTRY)

for q in ["Gold customers", "How many shipped orders?", "Revenue from Desk Lamp"]:
    r = agent.run(q)
    print(f"\nQ: {q}")
    print(f"  tables={r.plan.linked_tables} errors={r.validation_errors}")
    print(f"  {r.answer[:100]}...")

---

## 10. Ungrounded vs Grounded SQL Generation

In [ ]:
UNGROUNDED_SQL = "SELECT user_email, purchase_status FROM transactions WHERE id = 'ORD-1001'"
GROUNDED = agent.run("Status of ORD-1001")

print("Ungrounded (hallucinated tables/columns):")
print(" ", validate_grounded_sql(UNGROUNDED_SQL, ["orders"]))
print("\nGrounded plan SQL:")
print(" ", GROUNDED.plan.sql)
print(" ", GROUNDED.answer.split(chr(10))[0])

---

## 11. Join Path Discovery

Document FK paths so the planner knows how to connect tables.

In [ ]:
def join_paths_hint(fks: list[tuple[str, str, str, str]]) -> str:
    lines = ["# Join paths"]
    for src_t, src_c, dst_t, dst_c in fks:
        lines.append(f"- {src_t}.{src_c} = {dst_t}.{dst_c}")
    return "\n".join(lines)


print(join_paths_hint(FOREIGN_KEYS))

---

## 12. Eval Harness — Schema Grounding Quality

In [ ]:
EVAL_CASES: list[dict[str, Any]] = [
    {"q": "How many shipped orders?", "must_tables": ["orders"], "must_run": True},
    {"q": "ORD-1001 status", "must_tables": ["orders"], "must_run": True},
    {"q": "Gold tier emails", "must_tables": ["customers"], "must_run": True},
    {"q": "Desk Lamp units sold", "must_tables": ["order_items", "products"], "must_run": True},
]


def eval_grounding(agent: SchemaGroundedTextToSQL, cases: list[dict[str, Any]]) -> dict[str, Any]:
    passed = 0
    details: list[dict[str, Any]] = []
    for case in cases:
        r = agent.run(case["q"])
        tables_ok = all(t in r.plan.linked_tables for t in case["must_tables"])
        run_ok = (r.query is not None and not r.query.error) if case["must_run"] else True
        val_ok = not r.validation_errors
        ok = tables_ok and run_ok and val_ok
        passed += int(ok)
        details.append({"q": case["q"], "ok": ok, "tables": r.plan.linked_tables, "errors": r.validation_errors})
    return {"passed": passed, "total": len(cases), "details": details}


report = eval_grounding(agent, EVAL_CASES)
print(f"Grounding eval: {report['passed']}/{report['total']}")
for d in report["details"]:
    mark = "✓" if d["ok"] else "✗"
    print(f"  {mark} {d['q']} → tables={d['tables']}")

---

## 13. Schema Pruning Impact

Compare token-sized schema views for the same question.

In [ ]:
q = "Revenue from Desk Lamp line items"
full = schema_rich(SCHEMA_REGISTRY)
link = link_schema(q, SCHEMA_REGISTRY)
pruned = schema_rich(SCHEMA_REGISTRY, link.tables)
print(f"Full schema chars: {len(full)}")
print(f"Pruned schema chars: {len(pruned)} (tables={link.tables})")
print(f"Reduction: {100 - int(100 * len(pruned) / len(full))}%")

---

## 14. Common Grounding Failures

| Failure | Example | Fix |
|---------|---------|-----|
| Hallucinated table | `transactions` | Schema linking + validation |
| Wrong column name | `user_name` vs `name` | Rich schema with synonyms |
| Wrong enum value | `complete` vs `delivered` | Sample values in schema |
| Missing join | Query products without `order_items` | FK path hints + link expansion |
| Full schema overload | 500 tables in prompt | Prune to linked tables |
| No few-shot | Inconsistent SQL style | Schema-anchored examples |

---

## 15. Production Grounding Checklist

- [ ] Schema registry with descriptions, FKs, sample values
- [ ] Schema linking step before SQL generation
- [ ] Pruned schema view per question
- [ ] Few-shot examples using real identifiers
- [ ] Post-generation identifier validation
- [ ] SELECT-only execution with LIMIT
- [ ] Eval set for table linking + execution success
- [ ] Versioned schema snapshots tied to deployments

---

## 16. Optional Live LLM with Full Grounding Context

In [ ]:
def live_grounded_sql(question: str) -> str | None:
    if not USE_LIVE_LLM:
        return grounded_nl_to_sql(question, SCHEMA_REGISTRY).sql
    from langchain_openai import ChatOpenAI

    ctx = agent.prompt_context(question)
    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    raw = llm.invoke(ctx + "\n\nWrite one SQLite SELECT query only.").content.strip()
    return raw.strip("`").replace("sql", "").strip()


sql = live_grounded_sql("count gold customers")
print("SQL:", sql)
if sql:
    print("Validation:", validate_grounded_sql(sql, link_schema("gold customers", SCHEMA_REGISTRY).tables))
    print(execute_read_sql(DB, sql))

---

## 17. Quiz

1. What is schema linking and why does it matter for large databases?
2. Name three elements that belong in a rich schema registry beyond SQL types.
3. How do few-shot examples improve grounding?
4. What should post-generation validation check?
5. Why prune the schema view per question instead of sending the full warehouse DDL?

---

## 18. Summary

**Schema grounding** connects natural language to real database structure through linking, rich metadata, few-shot examples, and validation.

The ShopCo **SchemaGroundedTextToSQL** agent demonstrates:

- `link_schema()` → relevant tables + FK neighbors
- `schema_rich()` → pruned prompt view with samples and synonyms
- `grounded_nl_to_sql()` → SQL using real identifiers
- `validate_grounded_sql()` → block hallucinated columns
- Eval harness for grounding quality

Next: **SQL guardrails** for execution safety and **combining RAG + SQL** in unified support workflows.